### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [5]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the way humans do—they’re masters of **vocal mimicry**. Their ability to copy human speech (and other sounds) is a by‑product of several biological and social traits that evolved for very different purposes:\n\n| Reason | How it works in parrots | Why it shows up as “talking” to us |\n|--------|------------------------|-----------------------------------|\n| **Highly developed vocal learning** | Parrots belong to the order *Psittaciformes*, whose members have a brain region called the **song system** (similar to that of songbirds). This network lets them learn, store, and reproduce complex sounds throughout life. | Humans hear these copied sounds and interpret them as words, even though the bird is simply reproducing the acoustic pattern. |\n| **Social communication** | In the wild, parrots live in flocks and use a rich repertoire of calls to maintain bonds, signal danger, and coordinate movement. They are constantly listening to and echoing t

In [6]:
from langchain.tools import tool

@tool 
def get_weather(location:str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Boston using function. Use get_weather.', 'tool_calls': [{'id': 'fc_74d58493-4803-4a93-95e7-900ec81a951f', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 127, 'total_tokens': 169, 'completion_time': 0.089323489, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.006974783, 'prompt_tokens_details': None, 'queue_time': 0.211976894, 'total_time': 0.096298272}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a168da-0c27-7691-8037-c3c4b9ad4a37-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_74d58493-4803-4a93-95e7-900ec81a951f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'inp

### Tool Execution Loops

In [8]:
## Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

## Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated args
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

## Step 3: Pass results back to model for final responses
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# The current weateher in boston is 72f and sunny

It's sunny in Boston.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. We have function get_weather. Use it.', 'tool_calls': [{'id': 'fc_2a89d961-4b78-4c27-924f-e50591631742', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 126, 'total_tokens': 170, 'completion_time': 0.093495548, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.005975298, 'prompt_tokens_details': None, 'queue_time': 1.375715185, 'total_time': 0.099470846}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a168dd-bee5-70c2-9916-0cabed96794e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_2a89d961-4b78-4c27-92